In [4]:
   import os, shutil, pandas as pd
   ROOT = "/content/drive/MyDrive/rag_trust"
   cur, bak = f"{ROOT}/data/trustrag_flags.parquet", f"{ROOT}/data/trustrag_flags_before.parquet"
   n_old = len(pd.read_parquet(f"{ROOT}/data/features_before.parquet"))     # rows in the pre-rebuild build
   n_cur = len(pd.read_parquet(cur))
   print("pre-rebuild rows:", n_old, "| current flags file rows:", n_cur)
   if os.path.exists(bak):
       print("backup already exists:", len(pd.read_parquet(bak)), "rows")
   elif n_cur == n_old:
       shutil.copy(cur, bak); print("backup made")
   else:
       print("current flags file is NOT the pre-rebuild one and there is no backup: stop and tell me")

pre-rebuild rows: 25650 | current flags file rows: 25650
backup made


In [5]:
!pip install -q rouge-score
import os, glob, json
import numpy as np, pandas as pd
from sklearn.cluster import KMeans
from rouge_score import rouge_scorer
from google.colab import drive
drive.mount("/content/drive")

COS_T, ROUGE_T = 0.85, 0.25          # TrustRAG Stage-1 thresholds: verify against the appendix
hits = glob.glob("/content/drive/MyDrive/**/data/passages.jsonl", recursive=True)
print("found:", hits); ROOT = hits[0].rsplit("/data/", 1)[0]
out_dir = lambda ds: f"{ROOT}/step1" if ds == "nq" else f"{ROOT}/step1_{ds}"

df = pd.DataFrame([json.loads(l) for l in open(hits[0])])
g = df[df["context_id"].notna()].sort_values(["context_id", "rank"]).reset_index(drop=True)
G = len(g) // 5
assert len(g) % 5 == 0 and (g.groupby("context_id").size() == 5).all()

# same text convention as the feature cell: clean = title + text, poison and injection = text
DOCS = {ds: json.load(open(f"{out_dir(ds)}/docs.json")) for ds in g["dataset"].unique()}
def index_text(ds, source, pid, passage):
    if source == "clean":
        t = DOCS[ds][pid.split(":doc:", 1)[1]]["title"]
        return f"{t} {passage}" if t else passage
    return passage
g["itext"] = [index_text(*a) for a in zip(g.dataset, g.source, g.passage_id, g.passage)]

texts = json.load(open(f"{ROOT}/data/emb_texts.json")); E = np.load(f"{ROOT}/data/emb.npy")
idx = {t: i for i, t in enumerate(texts)}
missing = [t for t in g["itext"].unique() if t not in idx]
assert not missing, f"{len(missing)} texts have no cached embedding: run the feature cell on this file first"
X = E[g["itext"].map(idx).values].astype(np.float32)
X /= np.linalg.norm(X, axis=1, keepdims=True)
X = X.reshape(G, 5, -1); PASS = g["passage"].values.reshape(G, 5)

rs = rouge_scorer.RougeScorer(["rougeL"])
def mean_pair(members, fn):
    return float(np.mean([fn(a, b) for i, a in enumerate(members) for b in members[i + 1:]]))

removed = np.zeros((G, 5), bool)
for k in range(G):
    lab = KMeans(2, n_init=5, random_state=42).fit_predict(X[k])
    qual = []
    for c in (0, 1):
        m = np.where(lab == c)[0]
        if len(m) < 2:
            continue
        cos = mean_pair(list(X[k][m]), lambda a, b: float(a @ b))
        if cos < COS_T:
            continue
        rg = mean_pair([PASS[k][i] for i in m], lambda a, b: rs.score(a, b)["rougeL"].fmeasure)
        if rg >= ROUGE_T:
            qual.append((cos, m))
    if qual:
        removed[k, max(qual, key=lambda x: x[0])[1]] = True    # if both qualify, remove the higher-cosine one

g["trustrag_removed"] = removed.reshape(-1)
g[["context_id", "rank", "trustrag_removed"]].to_parquet(f"{ROOT}/data/trustrag_flags.parquet", index=False)

# calibration check (no tuning): how often does it remove anything on clean val n=0 groups? train n=1 as a sanity look
def share_any(mask):
    return g[mask].groupby("context_id")["trustrag_removed"].any().mean()
orig = g["group_variant"] == "original"
print("val n=0 groups where it removes anything:", round(float(share_any(orig & (g.split == "val") & (g.n_poison == 0))), 3))
t1 = g[orig & (g.split == "train") & (g.n_poison == 1)]
print("TRAIN n=1: poison removed", round(float(t1[t1.label == 1]["trustrag_removed"].mean()), 3),
      "| clean removed", round(float(t1[t1.label == 0]["trustrag_removed"].mean()), 3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
found: ['/content/drive/MyDrive/rag_trust/data/passages.jsonl']
val n=0 groups where it removes anything: 0.1
TRAIN n=1: poison removed 0.0 | clean removed 0.025


In [6]:
import pandas as pd
ROOT = "/content/drive/MyDrive/rag_trust"
old = pd.read_parquet(f"{ROOT}/data/trustrag_flags_before.parquet")
new = pd.read_parquet(f"{ROOT}/data/trustrag_flags.parquet")
m = old.merge(new, on=["context_id", "rank"], suffixes=("_old", "_new"), validate="one_to_one")
print(len(old), "old rows |", len(new), "new rows | old rows with a changed flag:",
      int((m["trustrag_removed_old"] != m["trustrag_removed_new"]).sum()))
S = pd.read_parquet(f"{ROOT}/data/scores.parquet", columns=["context_id", "rank", "group_variant"])
c = S.merge(new, on=["context_id", "rank"], how="left", validate="one_to_one")
print("rows in scores.parquet without a flag:", int(c["trustrag_removed"].isna().sum()), "of", len(c))
print(c[c["group_variant"].isin(["injection", "rewritten_clean"])].groupby("group_variant")["trustrag_removed"].count())

25650 old rows | 28650 new rows | old rows with a changed flag: 0
rows in scores.parquet without a flag: 0 of 28650
group_variant
injection           100
rewritten_clean    2900
Name: trustrag_removed, dtype: int64
